In [ ]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')
print(f'TensorLy tenalg backend: {tl.tenalg.get_backend()}')

In [ ]:
from moabb.paradigms import FilterBankMotorImagery, MotorImagery
from moabb.datasets import *
from hoda.tensorize import fh_power, fh_log_envelope
from hoda.classification import ZLogRatio, ZScore
dataset = AlexMI()
dataset.event_id




In [ ]:
events = list(dataset.event_id.keys())[:3]
events

In [ ]:
sfreq=250
paradigm = MotorImagery(events=events, n_classes=3, resample=sfreq)
paradigm.used_events(dataset)

In [ ]:
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[2],
     return_epochs=False
)


In [ ]:
import numpy as np
np.unique(y)

In [ ]:
X.shape

In [ ]:
import sys
sys.path.append('../')
from classification_mi import stf_transform 
from hoda.classification import ZScore
X_tfr_base = stf_transform(X)
X_tfr_base = ZScore().fit_transform(X_tfr_base)
X_tfr_base = tl.tensor(X_tfr_base)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
plt.plot(tl.to_numpy(X_tfr_base[0,1,4,:]))
plt.show()

In [ ]:
X_tfr_base.shape

In [ ]:
import plotly.express as px
import numpy as np

px.imshow(np.cov(tl.to_numpy(tl.unfold(X_tfr_base,1))))

In [ ]:
px.imshow(np.cov(tl.to_numpy(tl.unfold(X_tfr_base,2))))

In [ ]:
px.imshow(np.cov(tl.to_numpy(tl.unfold(X_tfr_base,3))))

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import scipy.stats as stats
import matplotlib
%matplotlib inline

measurements = np.random.normal(loc = 20, scale = 5, size=100)   
stats.probplot(tl.to_numpy(X_tfr_base).flatten(), dist="norm", plot=plt)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

x_mean = tl.to_numpy(X_tfr_base[y=='feet']).mean(axis=0)
for f in range(x_mean.shape[0]):
    sns.heatmap(x_mean[f], cmap='RdBu_r', center=0, square='True', vmin=-0.3, vmax=0.3)
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import pandas as pd

x = tl.to_numpy(X_tfr_base.flatten())
hist, bin_edges = np.histogram(x, bins=100000)
bins = (bin_edges[:-1] + bin_edges[1:]) / 2  # Compute the mean of each subsequent pair
df = pd.DataFrame({'bins':bins, 'count':hist})
px.histogram(df, x="bins", y="count")

In [ ]:
from hoda.hoda import HODA

hoda = HODA(
        rank=None,
        max_iter=1024,
        tol=1e-6,
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        forward=True,
        theta=0.2,
        refit_shrinkage=True,
)


In [ ]:
X_tfr_base = tl.tensor(X_tfr_base)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from hoda.util import solve_gevdh

plt.style.use('default')
%load_ext line_profiler

hoda.fit_backward(X_tfr_base,y)
df = pd.DataFrame(hoda.train_info_['backward'])
display(df)

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[0]))

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[1]))

In [ ]:
import plotly.express as px

px.imshow(tl.to_numpy(hoda.scatter_w_[2]))

In [ ]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

In [ ]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr', log_y=False)
    fig.show()

In [ ]:
px.line(df, x='iteration', y='update', log_y=True, color='mode')


In [ ]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [ ]:
px.line(df, x='flip', y='objective', log_y=False, color='mode')

In [ ]:
hoda.fit_forward(X_tfr_base,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

In [ ]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

In [ ]:
if hoda.extra_train_info:
    x.line(df, x='flip', y='mse', log_y=True)

In [ ]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [ ]:
for i in range(hoda.aps_[1].shape[1]):
    plt.plot(tl.to_numpy(hoda.aps_[1][:,i]))
    plt.show()

In [ ]:
for i in range(hoda.aps_[2].shape[1]):
    plt.plot(tl.to_numpy(hoda.aps_[2][:,i]))
    plt.show()

In [ ]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np

Xt = hoda.transform(X_tfr_base)
xt = tl.to_numpy(tl.unfold(Xt,0))

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

In [ ]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
x_viz = PCA(whiten=True, n_components=2).fit_transform(xt, y)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)
fig.update_layout(width=1000, height=800)

### 